In [1]:
1+1

2

In [2]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI


C:\Users\Admin\AppData\Local\Temp\ipykernel_1224\577953030.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
from langchain_community.llms import Ollama

# Connect to your local Llama 3.2 model
llm = Ollama(model="llama3.2")

response = llm.invoke("Explain what LangSmith does in simple terms.")
print(response)

C:\Users\Admin\AppData\Local\Temp\ipykernel_1224\3044438599.py:4: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.2")


LangSmith is a tool that helps you manage and analyze the relationships between different languages. It's often used by linguists, researchers, and language enthusiasts to study language structures, patterns, and variations.

Think of LangSmith like a big map that helps you navigate the connections between languages. It allows you to:

1. Compare languages side by side
2. Identify similarities and differences
3. Analyze language patterns and structures
4. Create charts and graphs to visualize the data

By using LangSmith, you can gain insights into how languages are related, how they've evolved over time, and how they're used in different contexts. It's a powerful tool for anyone interested in language and linguistics.


In [4]:
loader = TextLoader("Knowledge.txt")

In [5]:
document = loader.load()

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
docs = splitter.split_documents(document)

In [7]:
docs

[Document(metadata={'source': 'Knowledge.txt'}, page_content='Johnathan "John" Miles is a man defined by a meticulous and aspirational "love" for James Bond, a passion that borders on lifestyle emulation. By day, John is a 44-year-old risk analyst in Chicago, a profession steeped in data and predictability, which he actively contrasts with the sophisticated, crisis-driven world of 007. This fixation is not just casual fandom; it is a dedicated curation of his personal aesthetic and environment. He invests significantly in the specific trappings of the character, favoring navy blue, single-breasted suits, brown suede chukka boots, and a specific Omega Seamaster Diver 300M, which he acquired after his last bonus. His evenings are often spent perfecting the Vesper martiniâ€”having researched the exact non-mainstream ingredients like Lillet Blancâ€”or streaming the John Barry instrumental scores. For John, the "love" for Bond is an appreciation for an ideal of precision, classic masculinit

In [8]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\Admin\Desktop\Projects\BASIC_RAG_1\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1378.83it/s]


In [9]:
vectorstore = Chroma.from_documents(docs,embedding=embedding,persist_directory="./chroma_db")

In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

In [11]:
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant. Use the following content to answer the question.
                                        Context: {context}
                                        Question:{question}
                                          give a concise, factual answer.

""")

In [12]:
rag_chain=(
    {
        "context":retriever | (lambda docs:"\n\n".join([d.page_content for d in docs])),
        "question":RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()

)

In [13]:
#testing 1 and output of the model

In [14]:
query = rag_chain.invoke("who is jhon")    

In [15]:
query                                      

'Johnathan "John" Miles is a 44-year-old risk analyst in Chicago.'

In [16]:
#testing 2 and output of the model

In [17]:
query = rag_chain.invoke("Jhon is the fan of which fictional character")


In [18]:
query

'James Bond.'